Instalación de librerías para RAG local

In [1]:

%pip install -q langchain langchain-community langchain-huggingface chromadb sentence-transformers
%pip install -q transformers bitsandbytes accelerate
%pip install torch --force-reinstall --index-url https://download.pytorch.org/whl/cu121
%pip install -q pypdf
%pip install -q langchain-chroma
%pip install -q patool
%pip install -q peft
%pip install python-dotenv

print("¡Librerías importadas correctamente!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached https://download.pytorch.org/whl/markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached https://d

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.1.0 requires fsspec[http]<=2026.7.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
trl 1.17.1 requires torch>=2.6.0, but you have torch 2.5.1+cu121 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


... (6 outputs left out)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
¡Librerías importadas correctamente!



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Login

In [1]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

token = os.getenv("TOKEN")

login(token)

Descargar modelo si aún no tienes ./chroma_db_boe_completo

In [2]:
import os
from huggingface_hub import hf_hub_download
import patoolib

repo_id = "mroscardd/BOE_spain_2026"
filename = "chroma_db_boe_2026.rar"
destino = "./"

  

print(f"Descargando dataset desde Hugging Face...")
ruta_rar = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset",
    force_download=True
)

print('Dataset descargado')


os.makedirs(destino, exist_ok=True)

print("Descomprimiendo archivo RAR...")
patoolib.extract_archive(ruta_rar, outdir=destino)


if os.path.exists(ruta_rar):
    os.remove(ruta_rar)

print("¡Proceso completado! La base de datos vectorial desde el RAR está lista.")

    

Descargando dataset desde Hugging Face...


INFO patool: Extracting C:\Users\mroscardd\.cache\huggingface\hub\datasets--mroscardd--BOE_spain_2026\snapshots\ce245ce4445df90f0e0a0b5423e8495a2ca9248c\chroma_db_boe_2026.rar ...
INFO patool: could not find a 'file' executable, falling back to guess mime type by file extension
INFO patool: running 'C:\\Program Files\\WinRAR\\rar.EXE' 'x' '-kb' '-or' '--' 'C:\\Users\\mroscardd\\.cache\\huggingface\\hub\\datasets--mroscardd--BOE_spain_2026\\snapshots\\ce245ce4445df90f0e0a0b5423e8495a2ca9248c\\chroma_db_boe_2026.rar'


Dataset descargado
Descomprimiendo archivo RAR...


INFO patool: ... C:\Users\mroscardd\.cache\huggingface\hub\datasets--mroscardd--BOE_spain_2026\snapshots\ce245ce4445df90f0e0a0b5423e8495a2ca9248c\chroma_db_boe_2026.rar extracted to `./'.


¡Proceso completado! La base de datos vectorial desde el RAR está lista.


Cargar modelo en memoria

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# Cuantización a 4 bits (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True 
)

# Cargar Tokenizer y Modelo 
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # O el modelo que estés utilizando

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("¡Tokenizer y modelo cargados en memoria!")

Loading weights: 100%|██████████| 291/291 [00:10<00:00, 27.22it/s]


¡Tokenizer y modelo cargados en memoria!


Recuperar vectorStore

In [9]:

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'batch_size': 256, 'normalize_embeddings': True}
)


persist_directory = "./chroma_db_boe_completo"  

# Cargar la base de datos existente
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)


retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 8,
        "score_threshold": 0.35  # Filtra ruido y evita falsos positivos
    }
)

print(f"¡Base de datos cargada correctamente desde '{persist_directory}'!")



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8465.18it/s]


¡Base de datos cargada correctamente desde './chroma_db_boe_completo'!


Probar la búsqueda RAG + Inferencia con el modelo

In [10]:
from peft import PeftModel

adapter_path = "./adaptador_qlora"


model = PeftModel.from_pretrained(
    model,
    adapter_path
)

c:\Users\mroscardd\Desktop\Code\IA Keepcoding\proyectos\BOE_Model\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
c:\Users\mroscardd\Desktop\Code\IA Keepcoding\proyectos\BOE_Model\.venv\Lib\site-packages\peft\peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_att

In [11]:



pregunta_usuario = input("Introduce tu consulta")
docs_relacionados = retriever.invoke(pregunta_usuario)
contexto_leyes = "\n\n".join([f"--- Fragmento ---\n{doc.page_content}" for doc in docs_relacionados])

# Promp
messages = [
    {
        "role": "system",
        "content": (
            "Eres un asistente jurídico especializado en el ordenamiento jurídico español.\n"
            "Tu ÚNICA fuente de verdad es el CONTEXTO proporcionado.\n\n"
            "REGLAS STRICTAS:\n"
            "1. Responde a la pregunta utilizando la información explícita del CONTEXTO. "
        )
    },
    {
        "role": "user",
        "content": f"CONTEXTO LEGAL RECUPERADO:\n{contexto_leyes}\n\nPREGUNTA:\n{pregunta_usuario}"
    }
]

# Formateo
prompt_formateado = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

inputs = tokenizer(prompt_formateado, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,        
        temperature=0.6,       
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

# Decodificar la respuesta
input_length = inputs.input_ids.shape[1]
respuesta_limpia = tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

print(respuesta_limpia)



Sí, según el texto proporcionado, todos los ciudadanos tienen derecho a disfrutar de una vivienda digna y adecuada, en los términos dispuestos por la ley. Esto se establece en el artículo 8 del Código de la Vivienda del Estado, específicamente en la letra a) del apartado "Derechos del ciudadano en relación con la vivienda".
